# Persian AI-Text Detection — v2 evaluation layer on GPU (Colab)

This notebook completes the part of the v2 protocol that cannot run on the
project's CPU-only host: the **transformer** detectors, evaluated against the
split and metric protocol in `training/splits_v2.py`, `training/eval_metrics.py`
and `training/calibration.py`.

Why a separate notebook:

- the CPU host trains the classical models (TF-IDF/SVM/LogReg/NB, style-feature
  RF, length-only and perplexity-only baselines) for every condition;
- the fine-tuned transformers (ParsBERT here, plus the existing GPT-2-fa) are the
  models a GPU is actually needed for, and the review requires the same
  conditions, the same seeds and the same calibration for them;
- therefore this notebook **never redefines** the splits, the metrics or the
  calibrator: it imports them from the repo, so transformer numbers and CPU
  baseline numbers are directly comparable.

What it produces (all saved back to Drive, ready to paste into the report):

| artifact | content |
|---|---|
| `reports/transformer_v2.json` | every condition x model x seed result |
| `reports/tables/transformer_in_domain.md` | headline table, same shape as the CPU one |
| `reports/tables/transformer_conditions.md` | cross-generator / cross-domain / cross-style / few-shot |
| `reports/tables/transformer_attack.md` | clean vs paraphrased recall with paired CIs |
| `reports/transformer_errors/` | at least 100 FP and 100 FN for manual inspection |
| `trained/transformer_v2/<model>/` | checkpoint, tokenizer, calibration + metrics |

## 0. Runtime

In [ ]:
# Runtime > Change runtime type > T4 GPU before running anything else.
!nvidia-smi -L || echo "NO GPU: switch the runtime type first"

## 1. Install

`shekar` is only needed for the lemmatised CPU preprocessing path; the
transformers here use a lighter normalisation (`aitextd.text.normalize_for_pretrained`),
so it is optional. `sacrebleu`/`datasets` are not needed — the v2 dataset is a
single parquet file.

In [ ]:
!pip install -q "transformers>=4.44" "scikit-learn>=1.3" joblib pandas pyarrow
import torch, transformers, sklearn, numpy as np, pandas as pd
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Stage the repo and the v2 artifacts from Drive

Put this on your Drive first (the CPU host writes exactly these files):

```
MyDrive/AITextDetection/
├── aitextd/                       ← the whole package folder (dataset.py, text.py, paths.py, ...)
├── training/
│   ├── eval_metrics.py             ← the metric suite (imported, not reimplemented)
│   └── calibration.py              ← temperature/Platt/isotonic + the two-stage policy
├── data/
│   ├── dataset_v2.parquet          ← labels, domains, generators, topic groups, v1 membership
│   ├── splits_v2.joblib            ← every condition's train/val/test indices
│   ├── dataset_v2.manifest.json
│   ├── style_features_v2.parquet   ← (optional) the 20 style features
│   ├── ai_revised.parquet          ← (optional) machine-revised rows = the third class
│   └── paraphrase_attack.parquet   ← (optional) the attack set
└── models/
    └── gpt2-fa.mdl + gpt2_tokenizer/   ← (optional) the already fine-tuned GPT-2-fa
```

In [ ]:
import os, shutil, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/AITextDetection"
REPO = "/content/AITextDetection"
os.makedirs(REPO, exist_ok=True)

for folder in ("aitextd",):
    src, dst = f"{DRIVE_DIR}/{folder}", f"{REPO}/{folder}"
    if not os.path.isdir(src):
        raise SystemExit(f"missing {src}; upload the repo folder first")
    shutil.copytree(src, dst, dirs_exist_ok=True)
os.makedirs(f"{REPO}/training", exist_ok=True)
for name in ("eval_metrics.py", "calibration.py"):
    shutil.copy2(f"{DRIVE_DIR}/training/{name}", f"{REPO}/training/{name}")

os.makedirs(f"{REPO}/data", exist_ok=True)
for name in ("dataset_v2.parquet", "splits_v2.joblib"):
    src = f"{DRIVE_DIR}/data/{name}"
    if not os.path.exists(src):
        raise SystemExit(f"missing {src}")
    shutil.copy2(src, f"{REPO}/data/{name}")

OPTIONAL = ("style_features_v2.parquet", "ai_revised.parquet", "paraphrase_attack.parquet")
present = []
for name in OPTIONAL:
    src = f"{DRIVE_DIR}/data/{name}"
    if os.path.exists(src):
        shutil.copy2(src, f"{REPO}/data/{name}")
        present.append(name)
print("optional artifacts present:", present or "none")

sys.path.insert(0, f"{REPO}/training")
sys.path.insert(0, REPO)
import joblib
import eval_metrics as em
import calibration as cal
from aitextd import dataset as ds
from aitextd.text import normalize_for_pretrained
print("repo imports OK — metrics and calibration come from the repo, not from this notebook")

## 3. Load the dataset and the splits, and re-check the integrity assertions

In [ ]:
df = pd.read_parquet(f"{REPO}/data/dataset_v2.parquet").reset_index(drop=True)
splits = joblib.load(f"{REPO}/data/splits_v2.joblib")
ds.validate_frame(df)

print(df["label"].value_counts().to_string())
print(df["domain"].value_counts().to_string())
print(f"{len(df)} documents, {df['generator'].nunique()} machine sources")

# the notebook must not trust the split file blindly: re-verify the two
# properties the review cares about before any model is trained
tr = set(df.iloc[splits["in_domain_group__train"]]["topic_id"])
te = set(df.iloc[splits["in_domain_group__test"]]["topic_id"])
va = set(df.iloc[splits["in_domain_group__val"]]["topic_id"])
assert not (tr & te), f"topic leak train/test: {len(tr & te)}"
assert not (tr & va), f"topic leak train/val: {len(tr & va)}"
assert not (set(df.iloc[splits["in_domain_group__train"]]["doc_id"]) & set(df.iloc[splits["in_domain_group__test"]]["doc_id"]))
print("integrity checks passed: no topic or document is shared between the splits")

CONDITIONS = sorted({n[: -len("__train")] for n in splits if n.endswith("__train")})
print("conditions available:", CONDITIONS)

## 4. Tokenisation

One tokenised copy per model family (the tokenizers differ). Sequences are
truncated to 256 sub-word tokens — long enough for the median document, short
enough that a T4 finishes an epoch in minutes.

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MAX_LEN = 256
TEXTS = df["text"].tolist()
NORMALISED = [normalize_for_pretrained(t) for t in TEXTS]
LABELS = (df["label"].to_numpy() == ds.HUMAN).astype(int)  # 0 = machine, 1 = human

_tokenizers = {}
def tokenizer_for(name):
    if name not in _tokenizers:
        _tokenizers[name] = AutoTokenizer.from_pretrained(name)
    return _tokenizers[name]

class TextDataset(Dataset):
    def __init__(self, indices, tokenizer):
        self.indices = np.asarray(indices)
        enc = tokenizer(
            [NORMALISED[i] for i in self.indices],
            truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="np",
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = LABELS[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        return {
            "input_ids": torch.tensor(self.input_ids[position], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[position], dtype=torch.long),
            "labels": torch.tensor(self.labels[position], dtype=torch.long),
        }

print("tokeniser loaded; normalised", len(NORMALISED), "documents")

## 5. Training / evaluation engine

Design decisions, stated so they can be checked:

- **labels** are `0 = machine (ai + ai_revised)`, `1 = human`; probabilities are
  flipped to the project convention (`P(AI)`) before any metric is computed;
- **class weights** are balanced, because the human side is twice the machine
  side;
- **early stopping** on *validation* macro-F1 only — the test set is scored once,
  at the end of training, with the best checkpoint;
- **calibration** (temperature / Platt / isotonic, best on validation log-loss)
  is fitted on the validation split by the repo's `calibration.py`, then applied
  to the test scores; thresholds and the two-stage policy come from validation
  only;
- **CIs** are the repo's stratified bootstrap (`eval_metrics.bootstrap_ci`);
- seeds: pass 3 seeds for the headline table (the review asks for mean ± std).

In [ ]:
import time
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _ai_probabilities(model, tokenizer, indices, batch_size=64):
    model.eval()
    probabilities = np.zeros(len(indices), dtype=float)
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            chunk = indices[start : start + batch_size]
            enc = tokenizer(
                [NORMALISED[i] for i in chunk],
                truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt",
            ).to(DEVICE)
            logits = model(**enc).logits
            # class 0 = machine: flip to the project convention P(ai)
            probabilities[start : start + len(chunk)] = (1 - torch.softmax(logits, dim=-1)[:, 0]).cpu().numpy()
    return probabilities

def finetune_encoder(model_name, train_idx, val_idx, seed=42, epochs=3,
                     batch_size=32, grad_accum=1, lr=2e-5, max_len=MAX_LEN):
    torch.manual_seed(seed)
    np.random.seed(seed)
    tokenizer = tokenizer_for(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    train_ds = TextDataset(train_idx, tokenizer)
    val_ds = TextDataset(val_idx, tokenizer)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    counts = np.bincount(LABELS[train_idx], minlength=2)
    class_weights = torch.tensor(len(train_idx) / (2.0 * np.maximum(counts, 1)), dtype=torch.float32).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = max(1, len(train_loader) // grad_accum) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_macro_f1, best_state, history = -1.0, None, []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                loss = torch.nn.functional.cross_entropy(
                    model(**batch).logits, batch["labels"], weight=class_weights
                )
            scaler.scale(loss / grad_accum).backward()
            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            running += float(loss.item())
        val_probability = _ai_probabilities(model, tokenizer, val_idx)
        val_prediction = (val_probability >= 0.5).astype(int)
        val_macro_f1 = f1_score((LABELS[val_idx] == 1).astype(int), (val_prediction == 1).astype(int), average="macro", zero_division=0)
        history.append({"epoch": epoch + 1, "train_loss": running / max(len(train_loader), 1), "val_macro_f1": float(val_macro_f1)})
        print(f"  epoch {epoch + 1}/{epochs} loss={history[-1]['train_loss']:.4f} val macro-F1={val_macro_f1:.4f}", flush=True)
        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, tokenizer, {"history": history, "best_val_macro_f1": float(best_macro_f1)}

## 6. One condition, fully evaluated

`evaluate_condition` trains on `{condition}__train`, calibrates on the
condition's own validation split (falling back to the in-domain validation set
when a condition defines none), and scores `{condition}__test` with the full
metric suite. This mirrors `training/eval_suite.py::run_condition` exactly, so
the two tables can be printed side by side.

In [ ]:
def evaluate_condition(model_name, condition, seed=42, epochs=3, batch_size=32, grad_accum=1):
    train_idx = splits[f"{condition}__train"]
    val_idx = splits.get(f"{condition}__val", splits["in_domain_group__val"])
    test_idx = splits[f"{condition}__test"]

    started = time.time()
    model, tokenizer, training = finetune_encoder(
        model_name, train_idx, val_idx, seed=seed, epochs=epochs,
        batch_size=batch_size, grad_accum=grad_accum,
    )
    val_probability = _ai_probabilities(model, tokenizer, val_idx)
    y_val = (LABELS[val_idx] == 0).astype(int)          # 1 = machine
    calibrator, calibrator_name, calibration_report = cal.fit_best_calibrator(val_probability, y_val)

    test_probability = cal_transform = calibrator.transform(_ai_probabilities(model, tokenizer, test_idx)) if calibrator is not None else _ai_probabilities(model, tokenizer, test_idx)
    y_test = (LABELS[test_idx] == 0).astype(int)
    metrics = em.binary_metrics(y_test, test_probability)
    policy = cal.TwoStagePolicy().fit(y_val, val_probability)

    return {
        "model": model_name,
        "condition": condition,
        "seed": seed,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "calibrator": calibrator_name,
        "calibration_report": {k: float(v) for k, v in calibration_report.items() if isinstance(v, float)},
        "training": training,
        "metrics": metrics.to_dict(),
        "confusion": metrics.confusion,
        "ci95": em.bootstrap_ci(y_test, test_probability),
        "policy_test": policy.report(y_test, test_probability),
        "policy_validation": policy.val_report_,
        "seconds": round(time.time() - started, 1),
        "model_object": model,      # kept in memory only; stripped before saving
        "tokenizer_object": tokenizer,
    }

MODELS = [
    "HooshvareLab/bert-fa-base-uncased",   # ParsBERT: monolingual, strong on Persian
    "xlm-roberta-base",                    # multilingual control
]
SEEDS = [42, 43, 44]

Smoke test before the long run: one condition, one seed, one epoch, a small
slice. If this cell is green, the rest is time.

In [ ]:
_smoke = evaluate_condition(MODELS[0], "in_domain_group", seed=42, epochs=1, batch_size=16, grad_accum=4)
print("smoke macro-F1:", _smoke["metrics"]["metrics"]["macro_f1"], "| calibrator:", _smoke["calibrator"])
del _smoke
torch.cuda.empty_cache()

## 7. Headline conditions (in-domain, cross-generator, cross-style), 3 seeds each

In [ ]:
import copy, json, os
from tqdm.auto import tqdm

OUT_DIR = f"{REPO}/reports"
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
RESULTS_PATH = f"{OUT_DIR}/transformer_v2.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {"runs": [], "started_from": "colab_finetune_v2.ipynb"}

def save():
    json.dump(results, open(RESULTS_PATH, "w"), ensure_ascii=False, indent=1, default=str)
    shutil.copy2(RESULTS_PATH, f"{DRIVE_DIR}/trained/transformer_v2.json")

HEADLINE = ["in_domain_group", "cross_generator"] + sorted(
    n for n in [x[: -len("__train")] for x in splits] if n and n.startswith("cross_style__")
)
done = {(r["model"], r["condition"], r["seed"]) for r in results["runs"]}

for model_name in MODELS:
    for condition in HEADLINE:
        for seed in SEEDS:
            if (model_name, condition, seed) in done:
                print("cached", model_name, condition, seed); continue
            print(f"--- {model_name} | {condition} | seed {seed}", flush=True)
            outcome = evaluate_condition(model_name, condition, seed=seed)
            model = outcome.pop("model_object"); tokenizer = outcome.pop("tokenizer_object")
            results["runs"].append(outcome)
            save()
            # keep one checkpoint per (model, condition): the last seed's head
            dest = f"{DRIVE_DIR}/trained/transformer_v2/{model_name.split('/')[-1]}__{condition}__seed{seed}"
            model.save_pretrained(dest); tokenizer.save_pretrained(dest)
            del model
            torch.cuda.empty_cache()
print("done:", len(results["runs"]), "runs")

## 8. Cross-domain (held-out human domain) and few-shot

Cross-domain is the condition where a *new* human domain arrives: the model is
retrained without it and tested on the whole domain. Few-shot uses 8/32/128
labelled rows per class, which is the realistic "we only have a small labelled
sample" case.

In [ ]:
CROSS_DOMAIN = sorted(n[: -len("__train")] for n in splits if n.startswith("cross_domain__") and n.endswith("__train"))
FEW_SHOT = sorted(n[: -len("__train")] for n in splits if n.startswith("few_shot__") and n.endswith("__train"))
print("cross-domain:", CROSS_DOMAIN); print("few-shot:", FEW_SHOT)

for model_name in MODELS:
    for condition in CROSS_DOMAIN + FEW_SHOT:
        for seed in SEEDS[:1] if condition.startswith("few_shot__") else SEEDS:
            if (model_name, condition, seed) in done:
                continue
            outcome = evaluate_condition(model_name, condition, seed=seed)
            outcome.pop("model_object"); outcome.pop("tokenizer_object")
            results["runs"].append(outcome)
            save()
            print(f"{model_name} | {condition} | seed {seed}: macro-F1={outcome['metrics']['metrics']['macro_f1']}", flush=True)

## 9. Three-class head (human / ai / ai_revised) and the revision diagnostic

Requires `data/ai_revised.parquet` to have been merged into `dataset_v2.parquet`
by `training/build_dataset_v2.py` on the CPU host. This is the formulation the
review asks for as the primary problem statement, with the binary collapse
reported alongside.

In [ ]:
if ds.AI_REVISED not in set(df["label"]):
    print("no ai_revised rows in this dataset build — skipping the three-class head")
else:
    from transformers import AutoModelForSequenceClassification as _AMSC
    classes = [ds.HUMAN, ds.AI, ds.AI_REVISED]
    label_to_int = {c: i for i, c in enumerate(classes)}
    multi_labels = df["label"].map(label_to_int).to_numpy()
    train_idx = splits["in_domain_group__train"]
    test_idx = splits["in_domain_group__test"]

    for model_name in MODELS[:1]:
        for seed in SEEDS:
            tokenizer = tokenizer_for(model_name)
            model = _AMSC.from_pretrained(model_name, num_labels=3).to(DEVICE)
            train_ds = TextDataset(train_idx, tokenizer)
            train_ds.labels = multi_labels[np.asarray(train_idx)]
            loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
            optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
            for epoch in range(3):
                model.train()
                for batch in tqdm(loader, desc=f"3-class {model_name.split('/')[-1]} seed {seed} epoch {epoch+1}"):
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                        loss = torch.nn.functional.cross_entropy(model(**batch).logits, batch["labels"])
                    loss.backward(); optimizer.step(); optimizer.zero_grad()
            model.eval()
            probabilities = np.zeros((len(test_idx), 3))
            for start in range(0, len(test_idx), 64):
                chunk = np.asarray(test_idx)[start : start + 64]
                enc = tokenizer([NORMALISED[i] for i in chunk], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
                with torch.no_grad():
                    probabilities[start : start + len(chunk)] = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
            y_multi = multi_labels[np.asarray(test_idx)]
            record = {
                "model": f"{model_name}::3class", "condition": "three_class", "seed": seed,
                "multiclass": em.multiclass_metrics(y_multi, probabilities, classes).to_dict(),
                "binary_collapse": em.binary_metrics((y_multi != label_to_int[ds.HUMAN]).astype(int), 1 - probabilities[:, label_to_int[ds.HUMAN]]).to_dict(),
                "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
            }
            results["runs"].append(record)
            save()
            print(f"3-class seed {seed}: macro-F1={record['multiclass']['metrics']['macro_f1']:.4f}", flush=True)

## 10. Paraphrase attack (clean vs attacked, paired)

The attack set is generated on the CPU host (`training/paraphrase.py --mode
attack`) from AI test documents that no model ever trained on. The model here is
trained on the clean in-domain split only, then the attacked versions are
scored; the drop is reported with a paired bootstrap over documents, because the
same documents are measured twice.

In [ ]:
import numpy as np
attack_path = f"{REPO}/data/paraphrase_attack.parquet"
if not os.path.exists(attack_path):
    print("no paraphrase_attack.parquet — run training/paraphrase.py --mode attack on the CPU host first")
else:
    attack = pd.read_parquet(attack_path)
    source_ids = attack["extra"].map(lambda e: json.loads(e)["source_doc_id"]).tolist()
    id_to_position = {doc_id: position for position, doc_id in enumerate(df["doc_id"])}
    clean_idx = np.array([id_to_position[i] for i in source_ids if i in id_to_position])
    by_method = attack["revision_type"].to_numpy()

    for model_name in MODELS:
        for seed in SEEDS[:1]:
            model, tokenizer, _ = finetune_encoder(model_name, splits["in_domain_group__train"], splits["in_domain_group__val"], seed=seed)
            val_probability = _ai_probabilities(model, tokenizer, splits["in_domain_group__val"])
            calibrator, name, _ = cal.fit_best_calibrator(val_probability, (LABELS[splits["in_domain_group__val"]] == 0).astype(int))
            clean_probability = calibrator.transform(_ai_probabilities(model, tokenizer, clean_idx))
            clean_detected = (clean_probability >= 0.5).astype(float)

            # score the attacked texts through the same tokeniser path
            global NORMALISED
            offset = len(NORMALISED)
            NORMALISED = NORMALISED + [normalize_for_pretrained(t) for t in attack["text"].tolist()]
            attacked_idx = np.arange(offset, offset + len(attack))
            attacked_probability = calibrator.transform(_ai_probabilities(model, tokenizer, attacked_idx))
            attacked_detected = (attacked_probability >= 0.5).astype(float)

            record = {
                "model": model_name, "condition": "paraphrase_attack", "seed": seed,
                "calibrator": name,
                "clean_recall": float(clean_detected.mean()),
                "attacked_recall": float(attacked_detected.mean()),
                "paired_drop": em.paired_bootstrap_diff(clean_detected, attacked_detected),
                "by_method": {
                    method: {
                        "n": int((by_method == method).sum()),
                        "recall": float(attacked_detected[by_method == method].mean()),
                        "mean_probability": float(attacked_probability[by_method == method].mean()),
                    }
                    for method in sorted(set(by_method))
                },
            }
            results["runs"].append(record)
            save()
            print(f"attack {model_name}: clean {record['clean_recall']:.3f} -> attacked {record['attacked_recall']:.3f}", flush=True)
            NORMALISED = NORMALISED[:offset]
            del model
            torch.cuda.empty_cache()

## 11. Error analysis: at least 100 FP and 100 FN, saved for manual review

In [ ]:
os.makedirs(f"{REPO}/reports/transformer_errors", exist_ok=True)
model_name = MODELS[0]
seed = SEEDS[0]
outcome = evaluate_condition(model_name, "in_domain_group", seed=seed)
model = outcome["model_object"]; tokenizer = outcome["tokenizer_object"]
test_idx = splits["in_domain_group__test"]
val_probability = _ai_probabilities(model, tokenizer, splits["in_domain_group__val"])
calibrator, _, _ = cal.fit_best_calibrator(val_probability, (LABELS[splits["in_domain_group__val"]] == 0).astype(int))
probability = calibrator.transform(_ai_probabilities(model, tokenizer, test_idx))
y_machine = (LABELS[test_idx] == 0).astype(int)
subset = df.iloc[test_idx].copy()
subset["ai_probability"] = probability

for direction, frame in (
    ("fp", subset[y_machine == 0].sort_values("ai_probability", ascending=False).head(max(100, int((probability[y_machine == 0] >= 0.5).sum())))),
    ("fn", subset[y_machine == 1].sort_values("ai_probability").head(max(100, int((probability[y_machine == 1] < 0.5).sum())))),
):
    keep = ["doc_id", "label", "domain", "origin", "generator", "prompt_style", "length_words",
            "length_bucket", "formality", "ai_probability"]
    frame = frame[keep].copy()
    frame["text_excerpt"] = subset.loc[frame.index, "text"].str.slice(0, 400).str.replace(r"\s+", " ", regex=True)
    frame["review_verdict"] = ""; frame["review_notes"] = ""
    path = f"{REPO}/reports/transformer_errors/{direction}_review.csv"
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    shutil.copy2(path, f"{DRIVE_DIR}/trained/transformer_errors_{direction}.csv")
    print(direction, len(frame), "documents ->", path)
del outcome, model
torch.cuda.empty_cache()

## 12. Tables, and how they join the CPU results

In [ ]:
import numpy as np

def mean_std(values):
    values = [v for v in values if v is not None]
    return (float(np.mean(values)), float(np.std(values, ddof=1)) if len(values) > 1 else 0.0) if values else (float("nan"), float("nan"))

def table(condition_name, split_key=None):
    rows = [r for r in results["runs"] if r.get("condition") == condition_name and "metrics" in r]
    if not rows:
        return f"no runs yet for {condition_name}\n"
    by_model = {}
    for record in rows:
        by_model.setdefault(record["model"], []).append(record)
    lines = ["| model | seeds | n | macro-F1 (mean±sd) | recall AI | FPR@TPR95 | ROC-AUC | PR-AUC | ECE |",
             "|---|---|---|---|---|---|---|---|---|"]
    for model, records in sorted(by_model.items()):
        metrics = [r["metrics"]["metrics"] for r in records]
        cells = [model, str(len(records)), str(records[0]["n_test"])]
        for key in ("macro_f1", "recall_ai", "fpr_at_tpr95", "roc_auc", "pr_auc", "ece_width"):
            mean, std = mean_std([m.get(key) for m in metrics])
            cells.append(f"{mean:.4f} ± {std:.4f}" if key == "macro_f1" else f"{mean:.4f}")
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines) + "\n"

for condition_name, filename in (
    ("in_domain_group", "transformer_in_domain.md"),
    ("cross_generator", "transformer_cross_generator.md"),
    ("three_class", "transformer_three_class.md"),
    ("paraphrase_attack", "transformer_attack.md"),
):
    body = table(condition_name)
    open(f"{OUT_DIR}/tables/{filename}", "w").write(body)
    print(f"--- {condition_name} ---"); print(body)

attack_runs = [r for r in results["runs"] if r.get("condition") == "paraphrase_attack" and "clean_recall" in r]
if attack_runs:
    lines = ["| model | clean recall | attacked recall | paired drop (95% CI) | P(drop>0) |", "|---|---|---|---|---|"]
    for record in attack_runs:
        drop = record["paired_drop"]
        lines.append(f"| {record['model']} | {record['clean_recall']:.4f} | {record['attacked_recall']:.4f} | "
                     f"{drop['mean_difference']:.4f} [{drop['lo']:.4f}, {drop['hi']:.4f}] | {drop['p_difference_positive']:.3f} |")
    body = "\n".join(lines) + "\n"
    open(f"{OUT_DIR}/tables/transformer_attack.md", "w").write(body)
    print(body)

# the CPU baselines, for the side-by-side table in the report
cpu_path = f"{REPO}/reports/eval_results.json"
if os.path.exists(cpu_path):
    cpu = json.load(open(cpu_path))
    print("CPU baselines available for:", [c.get("condition") for c in cpu["conditions"]])
else:
    print("copy reports/eval_results.json from the CPU host to print the side-by-side table here")

shutil.copy2(f"{OUT_DIR}/transformer_v2.json", f"{DRIVE_DIR}/trained/transformer_v2.json")
print("everything saved to", f"{DRIVE_DIR}/trained/")

## 13. Acceptance checklist covered by this notebook

| requirement | where |
|---|---|
| same splits, no topic/generator/domain leak | section 3 re-verifies before training |
| calibration fitted on validation only | section 6 |
| thresholds and review policy from validation only | sections 6, 10 |
| ≥3 seeds with mean ± std | sections 7, 8 (`SEEDS`) |
| macro-F1, ROC-AUC, PR-AUC, FPR@TPR95, ECE, bootstrap CIs | `eval_metrics.py`, reported in section 12 |
| cross-generator (unseen LLMs) | section 7 |
| cross-domain (unseen human domain) | section 8 |
| few-shot (8/32/128 per class) | section 8 |
| machine-revised third class and its collapse | section 9 |
| paraphrase attack with paired CI | section 10 |
| ≥100 FP and ≥100 FN inspected | section 11 |
| honest limitations | the report's limitations chapter, fed by these CSVs |

**What this notebook does not do**: it does not claim that a high number proves
machine authorship of any specific document. Every table here is a population
estimate with an interval, produced under a fixed protocol; the report frames
the output as decision support that routes text to human review, not as proof.